# Phase 1 — Dataset Forensics & Exploratory Data Analysis

This notebook executes and visualizes dataset forensics for the Amazon ML Challenge entity resolution dataset. Per guidelines, all heavy logic imports from `src/`.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../code/business_entity_resolution'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.loading import load_tsv, parse_matched_ids
from src.evaluation.dataset_report import *

BASE = os.path.abspath('../Dataset')
print(f"Dataset root: {BASE}")


## 1. Inspect Table Scales and Schemas


In [ ]:
train_s1 = load_tsv(os.path.join(BASE, 'train', 'train_source1.tsv'))
print("Train Source 1 shape:", train_s1.shape)
train_s1.head()


## 2. Country Distribution & Distribution Shift Analysis

Note that test contains France, which is absent from train. The pipeline must not hardcode country branches.


In [ ]:
for split in ['train', 'test']:
    s1_path = os.path.join(BASE, split, f'{split}_source1.tsv')
    df = load_tsv(s1_path)
    print(f"\n--- {split.upper()} S1 Country Distribution ---")
    print(df['country'].value_counts(normalize=True).round(4) * 100)


## 3. Match Cardinality Analysis (Ground Truth)

Distribution of matches per S1 entity (singletons vs multi-matches).


In [ ]:
gt_df = load_tsv(os.path.join(BASE, 'train', 'train_ground_truth.tsv'))
gt_df['matches'] = gt_df['matched_entity_ids'].apply(lambda s: parse_matched_ids(s) if pd.notna(s) else [])
gt_df['match_count'] = gt_df['matches'].apply(len)

counts = gt_df['match_count'].value_counts().sort_index()
print(counts)

plt.figure(figsize=(8, 4))
counts.iloc[:10].plot(kind='bar', color='#4F46E5')
plt.title("Distribution of Matches per S1 Entity")
plt.xlabel("Number of Matches")
plt.ylabel("S1 Entities")
plt.grid(axis='y', alpha=0.3)
plt.show()


## 4. Key Takeaways for Blocking & Modeling
- Country is 100% consistent across true matches (hard blocking filter).
- Multi-match is common (avg ~3.5 matches per S1 entity).
- Strict singleton protection is required: false merges on singletons yield 0 score for that entity.
